# Recovery of $\Delta\eta = \eta_{\text{I}} - \eta_{\text{II}}$ under different boundary heights

Fix $\eta_{\text{I}} = 0.3$ and vary $\Delta\eta = \eta_{\text{I}} - \eta_{\text{II}}$ (so $\eta_{\text{II}} = 0.3 - \Delta\eta$).

Both subjects see the same stimuli ($r \in \{1,2,3\}$) and same fixation distribution,
but differ in boundary height:

- Population I: $a = 1.5$
- Population II: $a = 3$

For each $\Delta\eta$, estimate $(\eta_{\text{I}}, \eta_{\text{II}})$ independently for each subject,
then compute $\widehat{\Delta\eta} = \hat\eta_{\text{I}} - \hat\eta_{\text{II}}$.

**Expected result**: MLE $\widehat{\Delta\eta}$ lies on the identity line.
TADA $\widehat{\Delta\eta}$ is biased because the TADA pseudo-true $\eta$ depends on $a$,
even though $a$ is a nuisance parameter that the correct likelihood conditions on.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

from scipy.optimize import minimize, Bounds, LinearConstraint
from scipy.stats import chi2
import warnings
import time
import pickle
import glob

from efpt import aDDModel
from efpt.cython.batch import compute_addm_nll, compute_tada_mean_nll

import sys
sys.path.insert(0, "../..")
from shared import (extract_data, fit_subject, subject_sum_nll,
                     fit_equal_eta, greater_lrt, less_lrt, setup_latex_matplotlib)
setup_latex_matplotlib()

warnings.filterwarnings("ignore", message=r"delta_grad == 0\.0.*", category=UserWarning)

In [2]:
# Parameters
ETA_I = 0.3
SHARED_PARAMS = dict(kappa=0.5, sigma=1.0, b=0.0, x0=0.0)
A_I = 1.2
A_II = 3.5

# Delta values: eta_I - eta_II
DELTA_VALUES = np.linspace(-0.1, 0.1, 11)

# Same covariates for both subjects
R_RANGE = (1, 3)
GAMMA_SHAPE = 4.0
GAMMA_SCALE = 0.1

# Experiment settings
N_TRIALS = 2000
N_REPLICATIONS = 50
N_THREADS = -1

print(f"eta_I = {ETA_I} (fixed)")
print(f"delta = eta_I - eta_II in {DELTA_VALUES}")
print(f"eta_II = {ETA_I} - delta in [{ETA_I - DELTA_VALUES[-1]:.2f}, {ETA_I - DELTA_VALUES[0]:.2f}]")
print(f"Population I: a={A_I}, Population II: a={A_II}")
print(f"Both: r_range={R_RANGE}")
print(f"{N_TRIALS} trials/subject, {N_REPLICATIONS} replications per delta")

eta_I = 0.3 (fixed)
delta = eta_I - eta_II in [-0.1  -0.08 -0.06 -0.04 -0.02  0.    0.02  0.04  0.06  0.08  0.1 ]
eta_II = 0.3 - delta in [0.20, 0.40]
Population I: a=1.2, Population II: a=3.5
Both: r_range=(1, 3)
2000 trials/subject, 50 replications per delta


In [4]:
# Main loop
rng = np.random.default_rng(42)

results = {"mle": {}, "tada": {}}
for mode in ["mle", "tada"]:
    results[mode] = {d: [] for d in DELTA_VALUES}

start_time = time.time()
for delta in DELTA_VALUES:
    eta_II = ETA_I - delta
    model_I = aDDModel(eta=ETA_I, a=A_I, **SHARED_PARAMS)
    model_II = aDDModel(eta=eta_II, a=A_II, **SHARED_PARAMS)

    for rep in range(N_REPLICATIONS):
        exp_I = model_I.generate_experiment(
            n_trials=N_TRIALS, gamma_shape=GAMMA_SHAPE, gamma_scale=GAMMA_SCALE,
            r_range=R_RANGE, n_threads=N_THREADS, rng=rng,
        )
        exp_II = model_II.generate_experiment(
            n_trials=N_TRIALS, gamma_shape=GAMMA_SHAPE, gamma_scale=GAMMA_SCALE,
            r_range=R_RANGE, n_threads=N_THREADS, rng=rng,
        )
        data_I = extract_data(exp_I)
        data_II = extract_data(exp_II)

        for mode in ["mle", "tada"]:
            res_I = fit_subject(data_I, mode, n_threads=N_THREADS)
            res_II = fit_subject(data_II, mode, n_threads=N_THREADS)
            delta_hat = res_I.x[0] - res_II.x[0]
            results[mode][delta].append(delta_hat)

    elapsed = time.time() - start_time
    mle_mean = np.mean(results["mle"][delta])
    tada_mean = np.mean(results["tada"][delta])
    print(f"delta={delta:+.3f} (eta_II={eta_II:.3f}): "
          f"MLE delta_hat={mle_mean:+.4f}, TADA delta_hat={tada_mean:+.4f} "
          f"[{elapsed:.0f}s]")

print(f"\nTotal time: {time.time() - start_time:.0f}s")

# Save
fname = "varying_delta_boundary_result_" + time.strftime("%Y%m%d-%H%M%S") + ".pkl"
with open(fname, "wb") as f:
    pickle.dump({"DELTA_VALUES": DELTA_VALUES, "results": results,
                 "ETA_I": ETA_I, "A_I": A_I, "A_II": A_II,
                 "N_TRIALS": N_TRIALS, "N_REPLICATIONS": N_REPLICATIONS}, f)
print(f"Saved to {fname}")

delta=-0.100 (eta_II=0.400): MLE delta_hat=-0.1003, TADA delta_hat=-0.1877 [684s]
delta=-0.080 (eta_II=0.380): MLE delta_hat=-0.0778, TADA delta_hat=-0.1730 [1347s]
delta=-0.060 (eta_II=0.360): MLE delta_hat=-0.0628, TADA delta_hat=-0.1455 [2005s]
delta=-0.040 (eta_II=0.340): MLE delta_hat=-0.0360, TADA delta_hat=-0.1140 [2699s]
delta=-0.020 (eta_II=0.320): MLE delta_hat=-0.0132, TADA delta_hat=-0.1007 [3390s]
delta=+0.000 (eta_II=0.300): MLE delta_hat=+0.0031, TADA delta_hat=-0.0858 [4074s]
delta=+0.020 (eta_II=0.280): MLE delta_hat=+0.0228, TADA delta_hat=-0.0470 [4762s]
delta=+0.040 (eta_II=0.260): MLE delta_hat=+0.0443, TADA delta_hat=-0.0285 [5446s]
delta=+0.060 (eta_II=0.240): MLE delta_hat=+0.0681, TADA delta_hat=-0.0023 [6113s]
delta=+0.080 (eta_II=0.220): MLE delta_hat=+0.0973, TADA delta_hat=+0.0298 [6773s]
delta=+0.100 (eta_II=0.200): MLE delta_hat=+0.1070, TADA delta_hat=+0.0399 [7439s]

Total time: 7439s
Saved to varying_delta_boundary_result_20260523-053710.pkl


In [3]:
# Load results (if necessary)
result_files = sorted(glob.glob("varying_delta_boundary_result_*.pkl"))
with open(result_files[-1], "rb") as f:
    data = pickle.load(f)
DELTA_VALUES = data["DELTA_VALUES"]
results = data["results"]
print(f"Loaded from {result_files[-1]}")

Loaded from varying_delta_boundary_result_20260523-053710.pkl


In [4]:
# Plot: true delta vs estimated delta
fig, ax = plt.subplots(figsize=(8, 8), dpi=300)

ax.plot([-0.3, 0.3], [-0.3, 0.3], "k-", lw=1, label="identity function")

for mode, color, marker, label in [("mle", "blue", "o", rf"$\widehat{{\Delta \eta}}_{{10000}}^{{\mathrm{{ML}}}}$"),
                                    ("tada", "red", "x", rf"$\widehat{{\Delta \eta}}_{{10000}}^{{\mathrm{{TADA}}}}$")]:
    means = [np.mean(results[mode][d]) for d in DELTA_VALUES]
    stds = [np.std(results[mode][d]) for d in DELTA_VALUES]
    ax.errorbar(DELTA_VALUES, means, yerr=stds, fmt=marker, color=color,
                capsize=5, ms=8, lw=1.5, label=label)


ax.axhline(0, color="gray", ls=":", alpha=0.5)
ax.axvline(0, color="gray", ls=":", alpha=0.5)

ax.set_xlabel(r"True $\Delta\eta = \eta_{\text{I}} - \eta_{\text{II}}$", fontsize=20)
ax.set_ylabel(r"Estimated $\widehat{\Delta\eta} = \widehat\eta_{\text{I}} - \widehat\eta_{\text{II}}$", fontsize=20)
# ax.set_title(rf"$\eta_{\text{I}} = {ETA_I}$ fixed, Population I: $a={A_I}$, Population II: $a={A_II}$"
#              rf", $r \in {R_RANGE}$",
#              fontsize=12)
ax.legend(fontsize=15)
ax.tick_params(axis="both", labelsize=15)
ax.set_xlim(-0.12, 0.12)
ax.set_ylim(-0.26, 0.2)


xmin, xmax = ax.get_xlim()
ymin, ymax = ax.get_ylim()

# Shade quadrant II: x < 0, y > 0
ax.add_patch(Rectangle(
    (xmin, 0),          # lower-left corner
    0 - xmin,           # width
    ymax - 0,           # height
    facecolor="gray",
    alpha=0.12,
    edgecolor="none",
    zorder=0
))

# Shade quadrant IV: x > 0, y < 0
ax.add_patch(Rectangle(
    (0, ymin),
    xmax - 0,
    0 - ymin,
    facecolor="gray",
    alpha=0.12,
    edgecolor="none",
    zorder=0
))


plt.tight_layout()
plt.savefig("population_with_different_boundaries.png", bbox_inches="tight")
plt.show()

## One-sided LRT at $\Delta\eta = 0.02$

Truth: $\eta_{\text{I}} = 0.30$, $\eta_{\text{II}} = 0.28$, so $\eta_{\text{I}} > \eta_{\text{II}}$ and $H_0: \eta_{\text{I}} \le \eta_{\text{II}}$ is **false**.

From the plot above, TADA estimates $\widehat{\Delta\eta} \approx -0.04$ (wrong sign), so the one-sided test with TADA should have zero power — it cannot detect the true ordering.

We run the one-sided LRT over 200 replications:
- $\Lambda_+ = 0$ if $\hat\eta_{\text{I}} \le \hat\eta_{\text{II}}$ (sign check fails)
- $\Lambda_+ = 2(\text{NLL}_{\eta_{\text{I}}=\eta_{\text{II}}} - \text{NLL}_{\text{full}})$ otherwise
- Reject $H_0$ when $\Lambda_+ > \chi^2_{1, 0.90} = 2.706$

In [7]:
DELTA_TEST = 0.05
ETA_II_TEST = ETA_I - DELTA_TEST
ALPHA = 0.05
CRITICAL_ONE_SIDED = chi2.ppf(1 - 2 * ALPHA, df=1)
N_REPS_LRT = 200

# Main LRT loop
model_I = aDDModel(eta=ETA_I, a=A_I, **SHARED_PARAMS)
model_II = aDDModel(eta=ETA_II_TEST, a=A_II, **SHARED_PARAMS)
rng_lrt = np.random.default_rng(123)

greater_results = {"mle": [], "tada": []}

print(f"One-sided LRT: H0: eta_I <= eta_II  vs  H1: eta_I > eta_II")
print(f"Truth: eta_I={ETA_I}, eta_II={ETA_II_TEST} (delta={DELTA_TEST}), H0 is FALSE")
print(f"Critical value (alpha={ALPHA}): {CRITICAL_ONE_SIDED:.4f}")
print()

start_time = time.time()
for rep in range(N_REPS_LRT):
    exp_I = model_I.generate_experiment(
        n_trials=N_TRIALS, gamma_shape=GAMMA_SHAPE, gamma_scale=GAMMA_SCALE,
        r_range=R_RANGE, n_threads=N_THREADS, rng=rng_lrt,
    )
    exp_II = model_II.generate_experiment(
        n_trials=N_TRIALS, gamma_shape=GAMMA_SHAPE, gamma_scale=GAMMA_SCALE,
        r_range=R_RANGE, n_threads=N_THREADS, rng=rng_lrt,
    )
    dI = extract_data(exp_I)
    dII = extract_data(exp_II)

    for mode in ["mle", "tada"]:
        Lambda, reject, eta_I_hat, eta_II_hat = greater_lrt(
            dI, dII, mode, CRITICAL_ONE_SIDED, n_threads=N_THREADS)
        greater_results[mode].append({
            "Lambda": Lambda, "reject": reject,
            "eta_I": eta_I_hat, "eta_II": eta_II_hat,
        })

    if (rep + 1) % 20 == 0 or rep == 0:
        elapsed = time.time() - start_time
        mle_power = np.mean([r["reject"] for r in greater_results["mle"]])
        tada_power = np.mean([r["reject"] for r in greater_results["tada"]])
        print(f"Rep {rep+1:3d}/{N_REPS_LRT}: "
              f"MLE power={mle_power:.3f}, TADA power={tada_power:.3f} "
              f"[{elapsed:.0f}s]")

print(f"\nTotal time: {time.time() - start_time:.0f}s")

fname = "greater_lrt_delta002_" + time.strftime("%Y%m%d-%H%M%S") + ".pkl"
with open(fname, "wb") as f:
    pickle.dump(greater_results, f)
print(f"Saved to {fname}")

One-sided LRT: H0: eta_I <= eta_II  vs  H1: eta_I > eta_II
Truth: eta_I=0.3, eta_II=0.25 (delta=0.05), H0 is FALSE
Critical value (alpha=0.05): 2.7055

Rep   1/200: MLE power=0.000, TADA power=0.000 [57s]


Rep  20/200: MLE power=0.350, TADA power=0.100 [934s]
Rep  40/200: MLE power=0.350, TADA power=0.050 [2001s]
Rep  60/200: MLE power=0.383, TADA power=0.033 [3071s]
Rep  80/200: MLE power=0.362, TADA power=0.025 [4114s]
Rep 100/200: MLE power=0.340, TADA power=0.020 [5214s]
Rep 120/200: MLE power=0.333, TADA power=0.017 [6240s]
Rep 140/200: MLE power=0.350, TADA power=0.014 [7186s]
Rep 160/200: MLE power=0.362, TADA power=0.013 [8218s]
Rep 180/200: MLE power=0.361, TADA power=0.011 [9199s]
Rep 200/200: MLE power=0.360, TADA power=0.010 [10296s]

Total time: 10296s
Saved to greater_lrt_delta002_20260523-082847.pkl


In [8]:
# Load LRT results (if necessary)
greater_files = sorted(glob.glob("greater_lrt_delta002_*.pkl"))
with open(greater_files[-1], "rb") as f:
    greater_results = pickle.load(f)
print(f"Loaded from {greater_files[-1]}")

# Report
print(f"\nOne-sided LRT: H0: eta_I <= eta_II  vs  H1: eta_I > eta_II")
print(f"Truth: eta_I={ETA_I}, eta_II={ETA_II_TEST} (delta={DELTA_TEST}), H0 is FALSE")
print(f"alpha={ALPHA}, critical value={CRITICAL_ONE_SIDED:.4f}\n")

for mode in ["mle", "tada"]:
    etas_I = [r["eta_I"] for r in greater_results[mode]]
    etas_II = [r["eta_II"] for r in greater_results[mode]]
    rejections = [r["reject"] for r in greater_results[mode]]
    correct_dir = [r["eta_I"] > r["eta_II"] for r in greater_results[mode]]
    n_rep = len(rejections)
    print(f"{mode.upper():4s}:")
    print(f"  mean eta_I = {np.mean(etas_I):.4f}, mean eta_II = {np.mean(etas_II):.4f}")
    print(f"  correct direction (eta_I_hat > eta_II_hat): {sum(correct_dir)}/{n_rep} ({np.mean(correct_dir):.1%})")
    print(f"  reject H0 (power): {sum(rejections)}/{n_rep} ({np.mean(rejections):.1%})")
    print()

Loaded from greater_lrt_delta002_20260523-082847.pkl

One-sided LRT: H0: eta_I <= eta_II  vs  H1: eta_I > eta_II
Truth: eta_I=0.3, eta_II=0.25 (delta=0.05), H0 is FALSE
alpha=0.05, critical value=2.7055

MLE :
  mean eta_I = 0.2969, mean eta_II = 0.2474
  correct direction (eta_I_hat > eta_II_hat): 178/200 (89.0%)
  reject H0 (power): 72/200 (36.0%)

TADA:
  mean eta_I = 0.0553, mean eta_II = 0.0812
  correct direction (eta_I_hat > eta_II_hat): 65/200 (32.5%)
  reject H0 (power): 2/200 (1.0%)



## One-sided LRT for $H_0: \eta_{\text{I}} \ge \eta_{\text{II}}$ at $\Delta\eta = 0.02$: Type I error

Same truth: $\eta_{\text{I}} = 0.30$, $\eta_{\text{II}} = 0.28$. Now test the **opposite** direction:

$$H_0: \eta_{\text{I}} \ge \eta_{\text{II}} \qquad \text{vs} \qquad H_1: \eta_{\text{I}} < \eta_{\text{II}}$$

$H_0$ is **true** (since $0.30 \ge 0.28$). Rejections are **Type I errors**.

The one-sided LRT statistic is:
- $\Lambda_- = 0$ if $\hat\eta_{\text{I}} \ge \hat\eta_{\text{II}}$ (sign check fails)
- $\Lambda_- = 2(\text{NLL}_{\eta_{\text{I}}=\eta_{\text{II}}} - \text{NLL}_{\text{full}})$ if $\hat\eta_{\text{I}} < \hat\eta_{\text{II}}$
- Reject $H_0$ when $\Lambda_- > \chi^2_{1, 0.90} = 2.706$

TADA's boundary-dependent bias makes it estimate $\hat\eta_{\text{I}} < \hat\eta_{\text{II}}$, so the sign check passes and TADA **falsely rejects** — concluding population I has lower attentional discounting when it's actually higher.

In [9]:
# One-sided LRT for H0: eta_I >= eta_II
model_I_less = aDDModel(eta=ETA_I, a=A_I, **SHARED_PARAMS)
model_II_less = aDDModel(eta=ETA_II_TEST, a=A_II, **SHARED_PARAMS)
rng_less = np.random.default_rng(456)

less_results = {"mle": [], "tada": []}

print(f"One-sided LRT: H0: eta_I >= eta_II  vs  H1: eta_I < eta_II")
print(f"Truth: eta_I={ETA_I}, eta_II={ETA_II_TEST} (delta={DELTA_TEST}), H0 is TRUE")
print(f"Rejections are TYPE I ERRORS")
print(f"Critical value (alpha={ALPHA}): {CRITICAL_ONE_SIDED:.4f}")
print()

start_time = time.time()
for rep in range(N_REPS_LRT):
    exp_I = model_I_less.generate_experiment(
        n_trials=N_TRIALS, gamma_shape=GAMMA_SHAPE, gamma_scale=GAMMA_SCALE,
        r_range=R_RANGE, n_threads=N_THREADS, rng=rng_less,
    )
    exp_II = model_II_less.generate_experiment(
        n_trials=N_TRIALS, gamma_shape=GAMMA_SHAPE, gamma_scale=GAMMA_SCALE,
        r_range=R_RANGE, n_threads=N_THREADS, rng=rng_less,
    )
    dI = extract_data(exp_I)
    dII = extract_data(exp_II)

    for mode in ["mle", "tada"]:
        Lambda, reject, eta_I_hat, eta_II_hat = less_lrt(
            dI, dII, mode, CRITICAL_ONE_SIDED, n_threads=N_THREADS)
        less_results[mode].append({
            "Lambda": Lambda, "reject": reject,
            "eta_I": eta_I_hat, "eta_II": eta_II_hat,
        })

    if (rep + 1) % 20 == 0 or rep == 0:
        elapsed = time.time() - start_time
        mle_err = np.mean([r["reject"] for r in less_results["mle"]])
        tada_err = np.mean([r["reject"] for r in less_results["tada"]])
        print(f"Rep {rep+1:3d}/{N_REPS_LRT}: "
              f"MLE Type I error={mle_err:.3f}, TADA Type I error={tada_err:.3f} "
              f"[{elapsed:.0f}s]")

print(f"\nTotal time: {time.time() - start_time:.0f}s")

fname = "less_lrt_delta002_" + time.strftime("%Y%m%d-%H%M%S") + ".pkl"
with open(fname, "wb") as f:
    pickle.dump(less_results, f)
print(f"Saved to {fname}")

One-sided LRT: H0: eta_I >= eta_II  vs  H1: eta_I < eta_II
Truth: eta_I=0.3, eta_II=0.25 (delta=0.05), H0 is TRUE
Rejections are TYPE I ERRORS
Critical value (alpha=0.05): 2.7055

Rep   1/200: MLE Type I error=0.000, TADA Type I error=1.000 [10s]


Rep  20/200: MLE Type I error=0.000, TADA Type I error=0.250 [470s]
Rep  40/200: MLE Type I error=0.000, TADA Type I error=0.250 [898s]
Rep  60/200: MLE Type I error=0.000, TADA Type I error=0.250 [1243s]
Rep  80/200: MLE Type I error=0.000, TADA Type I error=0.225 [1681s]
Rep 100/200: MLE Type I error=0.000, TADA Type I error=0.200 [1970s]
Rep 120/200: MLE Type I error=0.000, TADA Type I error=0.183 [2318s]
Rep 140/200: MLE Type I error=0.000, TADA Type I error=0.186 [2720s]
Rep 160/200: MLE Type I error=0.000, TADA Type I error=0.181 [3122s]
Rep 180/200: MLE Type I error=0.006, TADA Type I error=0.183 [3590s]
Rep 200/200: MLE Type I error=0.005, TADA Type I error=0.180 [3918s]

Total time: 3918s
Saved to less_lrt_delta002_20260523-093406.pkl


In [10]:
# Load less_lrt results (if necessary)
less_files = sorted(glob.glob("less_lrt_delta002_*.pkl"))
with open(less_files[-1], "rb") as f:
    less_results = pickle.load(f)
print(f"Loaded from {less_files[-1]}")

# Report
print(f"\nOne-sided LRT: H0: eta_I >= eta_II  vs  H1: eta_I < eta_II")
print(f"Truth: eta_I={ETA_I} >= eta_II={ETA_II_TEST}, so H0 is TRUE")
print(f"alpha={ALPHA}, critical value={CRITICAL_ONE_SIDED:.4f}\n")

for mode in ["mle", "tada"]:
    etas_I = [r["eta_I"] for r in less_results[mode]]
    etas_II = [r["eta_II"] for r in less_results[mode]]
    rejections = [r["reject"] for r in less_results[mode]]
    wrong_dir = [r["eta_I"] < r["eta_II"] for r in less_results[mode]]
    n_rep = len(rejections)
    print(f"{mode.upper():4s}:")
    print(f"  mean eta_I = {np.mean(etas_I):.4f}, mean eta_II = {np.mean(etas_II):.4f}")
    print(f"  wrong direction (eta_I_hat < eta_II_hat): {sum(wrong_dir)}/{n_rep} ({np.mean(wrong_dir):.1%})")
    print(f"  reject H0 (Type I error): {sum(rejections)}/{n_rep} ({np.mean(rejections):.1%})")
    print()

Loaded from less_lrt_delta002_20260523-093406.pkl

One-sided LRT: H0: eta_I >= eta_II  vs  H1: eta_I < eta_II
Truth: eta_I=0.3 >= eta_II=0.25, so H0 is TRUE
alpha=0.05, critical value=2.7055

MLE :
  mean eta_I = 0.2970, mean eta_II = 0.2482
  wrong direction (eta_I_hat < eta_II_hat): 23/200 (11.5%)
  reject H0 (Type I error): 1/200 (0.5%)

TADA:
  mean eta_I = 0.0556, mean eta_II = 0.0818
  wrong direction (eta_I_hat < eta_II_hat): 132/200 (66.0%)
  reject H0 (Type I error): 36/200 (18.0%)



## Distribution of the one-sided LRT statistic $\Lambda_-$

Under $H_0: \eta_{\text{I}} \ge \eta_{\text{II}}$ (true, strictly), the MLE-based $\Lambda_-$ conditional on $\Lambda_- > 0$ should approximately follow $\chi^2_1$. Since $H_0$ holds strictly (not at the boundary), $P(\Lambda_- = 0)$ should be $> 50\%$ for MLE.

The TADA-based $\Lambda_-$ is a pseudo-LRT: the $\chi^2_1$ calibration does not hold because TADA is inconsistent. Its distribution should be shifted right (inflated), with fewer zeros (TADA's bias makes the sign check pass too often).

In [11]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x_grid = np.linspace(0.01, 15, 300)
# Theoretical: conditional on Lambda_- > 0, it should follow chi2_1
chi2_pdf = chi2.pdf(x_grid, df=1)

for ax, mode, title in zip(axes, ["mle", "tada"], ["MLE (correct)", "TADA (inconsistent)"]):
    lambdas = np.array([r["Lambda"] for r in less_results[mode]])
    rej_rate = np.mean([r["reject"] for r in less_results[mode]])
    n_zero = np.sum(lambdas == 0.0)
    n_total = len(lambdas)
    frac_zero = n_zero / n_total

    # Histogram of Lambda_- | Lambda_- > 0, normalized as density
    lambdas_pos = lambdas[lambdas > 0]
    if len(lambdas_pos) > 0:
        ax.hist(lambdas_pos, bins=25, density=True, alpha=0.7, color="steelblue",
                label=rf"$\Lambda_- \mid \Lambda_- > 0$ ({len(lambdas_pos)}/{n_total} reps)")

    # Theoretical reference
    ax.plot(x_grid, chi2_pdf, "r-", lw=2, label=r"$\chi^2_1$ density")

    ax.axvline(CRITICAL_ONE_SIDED, color="k", ls="--", lw=1.5,
               label=rf"critical value = {CRITICAL_ONE_SIDED:.2f}")

    ax.set_xlabel(r"$\Lambda_-$", fontsize=14)
    ax.set_ylabel("Density", fontsize=14)
    ax.set_title(f"{title}\n"
                 rf"$P(\Lambda_-=0)$: {frac_zero:.0%}, "
                 f"reject rate: {rej_rate:.1%} (nominal {ALPHA:.0%})",
                 fontsize=12)
    ax.legend(fontsize=10)
    ax.set_xlim(0, max(15, np.percentile(lambdas, 99) if lambdas.max() > 0 else 15))

plt.tight_layout()
plt.show()